# Test-Time Scaling Project

This notebook implements Best-of-N (Self-Consistency) inference for reasoning tasks.

Goal:
- Compare small model single-pass vs Best-of-N
- Evaluate accuracy vs compute tradeoff
- Optional: compare to larger teacher model


## 1. Install Dependencies

In [2]:
#!pip install transformers datasets accelerate bitsandbytes
!conda create -n ece226 python=3.10 -y

/bin/bash: line 1: conda: command not found


In [3]:
!conda activate ece226
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets accelerate bitsandbytes tqdm matplotlib ipykernel
!python -m ipykernel install --user --name ece226 --display-name "Python (ece226)"

/bin/bash: line 1: conda: command not found
Looking in indexes: https://download.pytorch.org/whl/cu121
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Installed kernelspec ece226 in /root/.local/share/jupyter/kernels/ece226


In [4]:
import torch
print("Torch imported")

Torch imported


In [5]:
#import torch
import sys

print("Python path:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

Python path: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA version: 12.8
CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.637086208


## 2. Load Dataset (GSM8K)

In [6]:
from datasets import load_dataset

dataset = load_dataset('gsm8k', 'main')
test_set = dataset['test']

print(test_set[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


## 3. Load Model (Small Model Recommended: 3B–8B)

Replace model_name with your chosen model.

In [7]:
#import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen3.5-2B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2048)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (in_proj_qkv): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (in_proj_z): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (in_proj_b): Linear4bit(in_features=2048, out_features=16, bias=False)
          (in_proj_a): Linear4bit(in_features=2048, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
  

In [8]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

True
Tesla T4
15.637086208 GB


## 4. Helper Functions

In [9]:
import re
from collections import Counter

def build_prompt(question):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": (f"{question}\n\n"
                                     "Do not include <think> tags in your response. keep your thinking process and response brief."
                "After solving, give the final answer in the format:\n"
                "Final Answer: <number>")}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )




def extract_gold(answer_text):
    match = re.search(r"####\s*([-+]?\d*\.\d+|\d+)", answer_text)
    if match:
        return match.group(1)
    return None


def extract_answer(text):
    # Find all integers or decimals (including negatives)
    numbers = re.findall(r"-?\d+\.?\d*", text)

    if not numbers:
        return None

    # Return the last number
    return numbers[-1]

"""
def extract_pred(text):
    match = re.search(r"Final Answer:\s*([-+]?\d*\.\d+|\d+)", text)
    if match:
        return match.group(1)
    return None
"""

def majority_vote(answers):
    counter = Counter(answers)
    return counter.most_common(1)[0][0]

def generate_answer(question, temperature=0.7):
    prompt = build_prompt(question)  # string

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=384,   # safer for 3060
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


<>:41: SyntaxWarning: invalid escape sequence '\s'
<>:41: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_4580/600413649.py:41: SyntaxWarning: invalid escape sequence '\s'
  match = re.search(r"Final Answer:\s*([-+]?\d*\.\d+|\d+)", text)


## 5. Best-of-N Implementation

In [10]:
def best_of_n(question, N=5):
    answers = []
    #prompt = question + "\nLet's think step by step."

    temp = 0
    if N > 1:
        temp = 0.7

    for _ in range(N):
        output = generate_answer(question, temp)
        print(output)
        ans = extract_answer(output)
        if ans:
            answers.append(ans)

    if len(answers) == 0:
        return None
    return majority_vote(answers)

## 6. Evaluation Loop

In [11]:
def evaluate(N=1, num_samples=100):
    correct = 0
    valid_samples = num_samples

    for i in range(num_samples):
        question = test_set[i]['question']
        gold = extract_gold(test_set[i]['answer'])
        print("Question:", question)
        print("Gold:", gold)

        if N == 1:
            output = generate_answer(question)
            pred = extract_answer(output)
        else:
            pred = best_of_n(question, N)

        if pred == gold and pred != None:
            correct += 1


        print("Pred:", pred)
        #print("Output:", output)
        print("-----")


    return correct / valid_samples

## 7. Run Experiments

Compare N = 1, 3, 5, 10

In [ ]:
for N in [1, 3, 5, 10]:
N = 1
acc = evaluate(N=N, num_samples=30)
print(f"N={N}, Accuracy={acc}")

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Gold: 18
Pred: 18
-----
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Gold: 3
Pred: 3
-----
Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
Gold: 70000
Pred: 150
-----
Question: James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?
Gold: 540
Pred: 180
-----
Question: Every day, Wendi feeds each of her chickens three cups of mixed chicken feed, containing seeds, mealworms and vegetables to help keep them healthy.  She gives 

In [ ]:
N = 3
acc = evaluate(N=N, num_samples=30)
print(f"N={N}, Accuracy={acc}")

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Gold: 18
system
You are a helpful assistant.
user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Do not include <think> tags in your response. keep your thinking process and response brief.After solving, give the final answer in the format:
Final Answer: <number>
assistant
<think>

</think>

1. Janet's ducks lay a total of 16 eggs per day.
2. She consumes 3 eggs for breakfast.
3. She uses 4 eggs to bake muffins.
4. The number of eggs remaining for sale is $16 - 3 - 4 = 

In [1]:
# N = 5
# acc = evaluate(N=N, num_samples=30)
# print(f"N={N}, Accuracy={acc}")

In [2]:
# N = 10
# acc = evaluate(N=N, num_samples=30)
# print(f"N={N}, Accuracy={acc}")

In [ ]:
#N = 20
#acc = evaluate(N=N, num_samples=30)
#print(f"N={N}, Accuracy={acc}")

## 8. Plot Accuracy vs N

In [3]:
# import matplotlib.pyplot as plt

# Ns = [1, 3, 5, 10]
# accuracies = [evaluate(N=n, num_samples=50) for n in Ns]

# plt.plot(Ns, accuracies)
# plt.xlabel('N (Number of Samples)')
# plt.ylabel('Accuracy')
# plt.title('Test-Time Scaling: Accuracy vs N')
# plt.show()